In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import os
import joblib
import warnings

warnings.filterwarnings('ignore')  # Suppress unnecessary warnings for cleaner output

In [2]:


# Define paths
data_dir = os.getcwd()
train_file = os.path.join(data_dir, 'panic_disorder_dataset_training.csv')
test_file = os.path.join(data_dir, 'panic_disorder_dataset_testing.csv')

# Load datasets
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)


In [3]:

# Handle missing values in categorical columns
missing_cols = ['Medical History', 'Psychiatric History', 'Substance Use']
for col in missing_cols:
    train_df[col] = train_df[col].fillna('Unknown')
    test_df[col] = test_df[col].fillna('Unknown')

# Drop unnecessary column
train_df = train_df.drop('Participant ID', axis=1)
test_df = test_df.drop('Participant ID', axis=1)

# Define categorical and numerical columns
categorical_cols = ['Gender', 'Family History', 'Personal History', 'Current Stressors', 
                    'Symptoms', 'Severity', 'Impact on Life', 'Demographics', 
                    'Medical History', 'Psychiatric History', 'Substance Use', 
                    'Coping Mechanisms', 'Social Support', 'Lifestyle Factors']
numerical_cols = ['Age']

# Prepare features and target
X_train = train_df.drop('Panic Disorder Diagnosis', axis=1)
y_train = train_df['Panic Disorder Diagnosis']
X_test = test_df.drop('Panic Disorder Diagnosis', axis=1)
y_test = test_df['Panic Disorder Diagnosis']

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),  # Normalize numerical features
        ('cat', 'passthrough', categorical_cols)   # Categorical will be encoded later
    ]
)


In [ ]:

# Encode categorical columns (after handling missing values)
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train[col], X_test[col]], axis=0)
    le.fit(combined.astype(str))  # Ensure string type for consistency
    X_train[col] = le.transform(X_train[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Build full ML pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))
])

# Hyperparameter grid for GridSearchCV to prevent overfitting
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
    'classifier__min_samples_split': [2, 5],
    'classifier__min_samples_leaf': [1, 2],
    'classifier__max_features': ['sqrt', 'log2']
}

# Stratified K-Fold for handling class imbalance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with cross-validation
grid_search = GridSearchCV(pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

# Best model
best_model = grid_search.best_estimator_

# Predict on test set
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

# Evaluation
print("\nBest Hyperparameters:")
print(grid_search.best_params_)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC AUC Score:")
print(roc_auc_score(y_test, y_pred_proba))

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': numerical_cols + categorical_cols,
    'Importance': best_model.named_steps['classifier'].feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nFeature Importance:")
print(feature_importance)

# Save the best model, preprocessor, and encoders
joblib.dump(best_model, 'optimized_panic_disorder_model.pkl')
joblib.dump(preprocessor, 'preprocessor.pkl')
for col, le in label_encoders.items():
    joblib.dump(le, f'label_encoder_{col}.pkl')